# Semantic Matching with Embeddings

**Learning Objectives**
  1. Learn how use Gemini embeddings with Google Gen AI SDK
  2. Learn how to calculate different simlarity metrics
  3. Learn how to identify matching products by using embeddings generated from descriptions
  4. Learn how to evaluate performance of a semantic matching
  
Semantic matching is the problem of classifying a pair of entities $(x_1, x_2)$ as being a good match or not. So it is a classification setup that is a very flexible: Namely, it comprises general information retrieval (where the first entity can be a textual query and the second entity can be a paragraph for instance), entity resolutions, or database-record fuzzy-matching. In this notebook we will focus on matching textual descriptions of retail products. More specifically:
  
  
**Use case description:** An online retail company scours the web to compare prices of products in their inventory with those offered by their competitors. Their first priority is to implement a model that compares the information on two product webpages and outputs a classification indicating whether the different product descriptions on the webpages correspond to an identical product, which we will refer to as a 'match'. We use the [Amazon-Google Products dataset](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) in the [entity resolution benchmark](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) created by Leipzig University. 

**Model description:** This notebook illustrates how to use Gemini API in Agent Platform to match product descriptions. The [Amazon-GoogleProducts dataset](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) contains product information about the products scrapped on Amazon or Google websites. It includes the products title, description, price, and manufacturer, athough this information is worded differently on the two websites. In this notebook, we will focus solely on building a model using the product titles. The idea is straightforward: we will create a prompt that asks the Gemini API whether the product titles match or not. Although the information about the products is limited to these titles, we still achieve an accuracy close to 100% on the test set.

## Setup

In [ ]:
import random
import os
import numpy as np
import pandas as pd
from google import genai
import os
from google import genai
from google.genai import types
from google.genai.types import GenerateContentConfig

pd.options.display.max_colwidth = 1000

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = f"{PROJECT}-cord19-semantic-search"

# Do not change these
os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

### Text embeddings API
The Text embeddings API offers models that convert text into numerical vectors to capture semantic meaning and context. 
The recommended model is gemini-embedding-001, which delivers state-of-the-art performance across English, multilingual, and code tasks with output dimensions up to 3072. For more specific use cases, text-embedding-005 is specialized for English and code tasks, while text-multilingual-embedding-002 focuses on multilingual applications. All three models support a maximum sequence length of 2048 tokens
More details:
https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/text-embeddings-api

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001" 
#EMBEDDING_MODEL = "text-embedding-005"
#EMBEDDING_MODEL = "text-multilingual-embedding-002"

### Text Embedding Task Types
The following table describes the task_type parameter values and their use cases:

| `Task type` | Description |
|---|---|
| **`RETRIEVAL_QUERY`** | Specifies the given text is a query in a search or retrieval setting. Use `RETRIEVAL_DOCUMENT` for the document side. |
| **`RETRIEVAL_DOCUMENT`** | Specifies the given text is a document in a search or retrieval setting. |
| **`SEMANTIC_SIMILARITY`** | Specifies the given text is used for Semantic Textual Similarity (STS). |
| **`CLASSIFICATION`** | Specifies that the embedding is used for classification. |
| **`CLUSTERING`** | Specifies that the embedding is used for clustering. |
| **`QUESTION_ANSWERING`** | Specifies that the query embedding is used for answering questions. Use `RETRIEVAL_DOCUMENT` for the document side. |
| **`FACT_VERIFICATION`** | Specifies that the query embedding is used for fact verification. Use `RETRIEVAL_DOCUMENT` for the document side. |
| **`CODE_RETRIEVAL_QUERY`** | Specifies that the query embedding is used for code retrieval for Java and Python. Use `RETRIEVAL_DOCUMENT` for the document side. |

---

### 🔍 Retrieval Tasks

* **Query**: Use `task_type=RETRIEVAL_QUERY` to indicate that the input text is a search query.
* **Corpus**: Use `task_type=RETRIEVAL_DOCUMENT` to indicate that the input text is part of the document collection being searched.

### Similarity Tasks

* **Semantic similarity**: Use `task_type=SEMANTIC_SIMILARITY` for both input texts to assess their overall meaning similarity.

In [ ]:
EMBD_TASK_TYPE="SEMANTIC_SIMILARITY"

### Initialize the client:
using Google GenAI client with parameter vertexai=True and provide Google Cloud Project ID and Region

In [ ]:
client = genai.Client(
    vertexai=True,
    project=PROJECT,
    location="global"
)

## Generate the embedding
### Define utility method to create embedding from text string:

In [ ]:
def embed_text(text_to_embed: str | list[str], output_dimensionality: int = None) -> list[list[float]]:
    """
    Generates embeddings for the provided text.
    """
    try:
        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=text_to_embed,
            config=types.EmbedContentConfig(
                task_type=EMBD_TASK_TYPE,
                output_dimensionality=output_dimensionality 
            )
        )
        
        return [np.array(embed.values) for embed in response.embeddings]

    except Exception as e:
        print(f"Failed to generate embeddings: {e}")

### Example on how to use embed_text method:

In [ ]:
text_to_embed = "emc retrospect 7.5 disk to diskcromwindows"

test_embedding = embed_text(text_to_embed)
print(f"Generated a vector with {len(test_embedding[0])} dimensions.")
print(f"View the first 5 floats: {test_embedding[0][:5]}") # View the first 5 floats
input_embd_vector=test_embedding[0]

### Embedding Vector Normalization (L2 Norm)

Lets calculate L2 norm (Euclidean magnitude) using `np.linalg.norm()`. As you can see the resulting vector magnitude is approximately `1.0`, confirming that the embeddings are unit-normalized. Normalized embeddings simplify downstream tasks because the cosine similarity between two vectors can be computed directly using a simple dot product.

In [ ]:
magnitude = np.linalg.norm(input_embd_vector)
print(f"vector magnitude = {magnitude}")

### L2 Normalization for MLR Embeddings
**Important Normalization Rule**: The API automatically L2-normalizes the default size output vector. However, if you use output_dimensionality to request a smaller size (like 128 or 768), the API returns raw, unnormalized numbers. You must manually normalize the truncated vector in your code if you wnat to use dot product instead of cosine similarity.

#### Option 1. Use output_dimensionality parameter:

In [ ]:
embedding_mrl_128 = embed_text(text_to_embed, output_dimensionality=128)
print(f"Generated a vector with {len(embedding_mrl_128[0])} dimensions.")
magnitude = np.linalg.norm(embedding_mrl_128)
print(f"vector magnitude = {magnitude}")

#### Option 2. Just trancate it:

In [ ]:
embedding_trunc_128 = input_embd_vector[:128]
print(f"Generated a vector with {len(embedding_trunc_128)} dimensions.")
magnitude = np.linalg.norm(embedding_trunc_128)
print(f"vector magnitude = {magnitude}")

### Embeddings L2 Normalization

**Vector $L_2$ normalization** scales a non-zero vector so that its total Euclidean length (magnitude) is exactly $1$. This process transforms any vector into a **unit vector** pointing in the exact same direction. 

In machine learning and embedding workflows, normalizing vectors to unit length is a very common preprocessing step. When vectors are $L_2$ normalized, calculating their **Cosine Similarity** simplifies down to a simple, computationally efficient **Dot Product**.
The formula for $L_2$ normalization divides a vector $v$ by its Euclidean ($L_2$) norm:

$$\hat{v} = \frac{v}{\|v\|_2}$$

#### 1. Calculating the $L_2$ Norm ($\|v\|_2$)
The denominator is the Euclidean length, calculated as the square root of the sum of the squared elements:

$$\|v\|_2 = \sqrt{\sum_{i=1}^{n} v_i^2} = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}$$

#### 2. Element-Wise Division
Once the scalar norm is found, every element in the vector is divided by it:

$$\hat{v} = \left[ \frac{v_1}{\|v\|_2}, \frac{v_2}{\|v\|_2}, \dots, \frac{v_n}{\|v\|_2} \right]$$

#### ⚙️ Implementation of L2 Normalization

The basic implementation of this function is straightforward with NumPy:

In [ ]:
def l2_normalize(vec):
    """
    Normalizes a vector to unit length (L2 Norm = 1).
    """
    # Calculate the L2 norm
    norm = np.linalg.norm(vec)
    if norm == 0:
        # Division-by-Zero Protection
        return vec
    return vec / norm

#### Test L2 Normalization implementation

In [ ]:
# Test 1: Standard 3D vector (norm of [3, 4] is 5, so normalized is [0.6, 0.8])
res1 = l2_normalize([3.0, 4.0])
assert np.allclose(res1, [0.6, 0.8]), f"Failed Standard: Expected [0.6, 0.8], got {res1}"

# Test 2: Unit Vector check (norm of any successfully normalized vector must be exactly 1)
assert np.isclose(np.linalg.norm(res1), 1.0), "Failed Unit Length check"

# Test 3: Zero Vector Safety (should return zeros without raising ZeroDivisionError)
res2 = l2_normalize([0.0, 0.0])
assert np.allclose(res2, [0.0, 0.0]), f"Failed Zero Safety: Expected [0.0, 0.0], got {res2}"

# Test 4: Already Normalized Vector (should remain unchanged)
res3 = l2_normalize([1.0, 0.0])
assert np.allclose(res3, [1.0, 0.0]), f"Failed Already Normalized: Expected [1.0, 0.0], got {res3}"

print("✅ All quick asserts passed successfully!")

Chec vector magnitude after L2 normalization:

In [ ]:
embedding_l2_norm_mrl_128 = l2_normalize(embedding_trunc_128)
magnitude = np.linalg.norm(embedding_l2_norm_mrl_128)
print(f"vector magnitude = {magnitude}")

### Visualize embedding:

In [ ]:
import plotly.express as px
import numpy as np

# Plot the vector as a sequence of bars
fig = px.bar(
    x=range(len(input_embd_vector)),
    y=input_embd_vector,
    # Crucial change: pass the values to 'color' to map them to a continuous scale
    color=input_embd_vector, 
    # Use a diverging color scale (Red for positive, Blue for negative)
    color_continuous_scale="RdBu_r", 
    labels={"x": f"Vector Dimension (0 - {len(input_embd_vector)})", "y": "Activation Value", "color": "Value"},
    title="Vector Visualization (Color-Coded Activation)",
    template="plotly_white"
)

# Style the bars to be sharp and distinct
fig.update_traces(marker_line_width=0)
fig.update_layout(
    bargap=0.1,
    coloraxis_cmid=0 
)

fig.show()

### Load first dataset with product descriptions

We use a [dataset of product information](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) scraped from Google and Amazon websites. The dataset is part of a [benchmark for semantic matching and entity resolution](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) from Leipzig University. It contains 3 tables which are included in this repo:

```python
../data/Amazon.csv.gz 
../data/GoogleProducts.csv.gz
../Amzon_GoogleProducts_perfectMapping.csv.gz

```

The first table contains product information listed on Amazon, including the product title, description, and manufacturer:

In [ ]:
amazon = pd.read_csv("../data/Amazon.csv.gz")
amazon.columns = [
    "idAmazon",
    "amazon_title",
    "amazon_description",
    "amazon_manufacturer",
    "amazon_price",
]
amazon = amazon.dropna(subset=['amazon_description'])
amazon.head()

### Lets create embeddings for concatenated title and description with a space in between

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter
import pandas as pd

# 1. Define your batch size
BATCH_SIZE = 20

# Concatenate Title and Description with a space in between
combined_text = amazon['amazon_title'].fillna('') + " " + amazon['amazon_description'].fillna('')

# Convert to list and apply the truncation logic we used earlier
MAX_CHARS = 4000
descriptions = [str(text)[:MAX_CHARS] for text in combined_text.tolist()]

all_embeddings = []

print(f"Starting to embed {len(descriptions)} documents in batches of {BATCH_SIZE}...")

# 3. Loop through the list in chunks
for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # Slice the list to get the current batch
    batch_texts = descriptions[i : i + BATCH_SIZE]
    
    # Call embed_text function
    batch_embeddings = embed_text(batch_texts)
    
    # Safety check: if the API call failed and returned an empty list, 
    # pad with None so our final list length matches the DataFrame length.
    if not batch_embeddings:
        print(f"Batch {i} to {i + BATCH_SIZE} failed. Padding with None.")
        batch_embeddings = [None] * len(batch_texts)
        
    all_embeddings.extend(batch_embeddings)
    
    # Pause briefly to avoid hitting Requests-Per-Minute (RPM) limits.
    # Adjust this value based on your specific API quota.
    time.sleep(1.0) 

# 4. Assign the resulting embeddings back to the DataFrame
if len(all_embeddings) == len(amazon):
    amazon['amazon_embedding'] = all_embeddings
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(batch_embeddings)} embeddings for {len(amazon)} rows.")

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

## Implement embedding alignment mearures
### ⚙️ Implementation of Dot Product
The **dot product** (also known as the scalar product) of two $n$-dimensional vectors, $A$ and $B$, is calculated by multiplying corresponding elements from each vector and then summing those products together.

$$A \cdot B = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + \dots + a_n b_n$$

Where:
* $a_i$ and $b_i$ represent the individual components (elements) of vectors $A$ and $B$ at index $i$.
* $n$ is the total number of dimensions (length of the embedding).

In [ ]:
def calculate_dot_product(vec1, vec2):
    """
    Calculates the Dot Product between two embedding vectors.
    """
    # Zip corresponding elements, multiply them, and sum the results
    #return sum(a * b for a, b in zip(vec1, vec2))
    return np.dot(vec1, vec2)

### Test Dot Product implementation

In [ ]:
# Test 1: Orthogonal
assert calculate_dot_product([1, 0], [0, 1]) == 0, "Failed Orthogonal Test"
# Test 2: Standard
assert calculate_dot_product([1, 2, 3], [4, 5, 6]) == 32, "Failed Standard Test"
# Test 3: Negatives
assert calculate_dot_product([2, 4, -1], [3, -2, 5]) == -7, "Failed Negative Test"
print("✅ All quick tests passed successfully!")

### ⚙️ Implementation of cosine similarity

**Cosine similarity** is a metric used to measure how similar two vectors are, irrespective of their size. It measures the cosine of the angle between two vectors projected in a multi-dimensional space. 

* 📌 **Key Application:** Widely used in Natural Language Processing (NLP) and Recommendation Systems to compare document or text embeddings.
* ⚠️ **Value Range:** It outputs a value between **-1 and 1**, where **1** means the vectors are identical in direction, **0** means they are orthogonal (perpendicular), and **-1** means they are diametrically opposed.

---

#### Mathematical Formula

The cosine similarity between two vectors $\mathbf{A}$ and $\mathbf{B}$ is calculated as:

$$\text{Cosine Similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

Where:
* $\mathbf{A} \cdot \mathbf{B}$ is the **dot product** of the vectors.
* $\|\mathbf{A}\|$ and $\|\mathbf{B}\|$ are the **Euclidean norms (magnitudes)** of the vectors.

---

#### Code Breakdown

Below is an overview of how each component of the `calculate_cosine_similarity` function maps to the mathematical formula:

| Step / Line | Code Segment | Mathematical Equivalent | Purpose / Explanation |
| :--- | :--- | :--- | :--- |
| **1. Dot Product** | `np.dot(embedding1, embedding2)` | $\mathbf{A} \cdot \mathbf{B}$ | Computes the sum of the products of the corresponding entries of the two sequences of numbers. |
| **2. Magnitudes** | `norm(embedding1) * norm(embedding2)` | $\|\mathbf{A}\| \times \|\mathbf{B}\|$ | Calculates the product of the L2 norms (Euclidean lengths) of both vectors. |
| **3. Safety Check** | `if magnitude == 0: return 0.0` | $N/A$ | **Avoids division by zero** errors if one of the input embeddings is an all-zero vector. |
| **4. Final Division** | `return dot_product / magnitude` | $\frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$ | Divides the dot product by the product of the magnitudes to get the final cosine similarity score. |

---

#### Code Implementation

In [ ]:
from numpy.linalg import norm
def calculate_cosine_similarity(embedding1, embedding2):
    # Calculate the dot product and the magnitudes
    dot_product = np.dot(embedding1, embedding2)
    magnitude = norm(embedding1) * norm(embedding2)
    
    # Avoid division by zero in case of an empty vector
    if magnitude == 0:
        return 0.0

    # Final Division
    return dot_product / magnitude

### Test Cosine Similarity implementation

In [ ]:
# 1. Identical
assert abs(calculate_cosine_similarity([3, 4], [3, 4]) - 1.0) < 1e-9

# 2. Opposite
assert abs(calculate_cosine_similarity([1, -1], [-1, 1]) - (-1.0)) < 1e-9

# 3. Orthogonal
assert abs(calculate_cosine_similarity([1, 0], [0, 1]) - 0.0) < 1e-9

# 4. Zero Safety
assert calculate_cosine_similarity([0, 0], [1, 2]) == 0.0

print("✅ All quick checks passed!")

### ⚙️ Implement Euclidian distance
The **Euclidean distance** is the most common metric used to calculate the straight-line distance between two points in a Euclidean space. 

In machine learning and natural language processing, it is used to measure the dissimilarity between two vector embeddings. A distance of `0` means the embeddings are identical, and larger values indicate greater dissimilarity.

---

#### Mathematical Formula

For two $n$-dimensional vectors, $A$ and $B$, the Euclidean distance is defined mathematically as:

$$d(A, B) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

#### Step-by-Step Breakdown:
1. **Element-Wise Subtraction ($a_i - b_i$):** Find the difference between corresponding dimensions of the two vectors.
2. **Squaring the Differences ($(a_i - b_i)^2$):** Square each resulting difference. This eliminates negative signs and penalizes larger differences more heavily.
3. **Summation ($\sum$):** Sum all the squared differences together.
4. **Square Root ($\sqrt{x}$):** Take the square root of the sum to bring the scale of the distance back to the original unit dimensions.

---

#### Implement Euclidian distance:

In [ ]:
def calculate_euclidean_distance(embedding1, embedding2):
    """
    Calculates the Euclidean distance between two embedding vectors.
    
    Formula: sqrt(sum((A_i - B_i)^2))
    """
    distance = np.sqrt(np.sum((embedding1 - embedding2) ** 2))
    return distance

### Test Euclidean distance implementation

In [ ]:
# 1. Identity
assert calculate_euclidean_distance(np.asarray([1, 2, 3]), np.asarray([1, 2, 3])) == 0.0

# 2. Known 3D Distance (sqrt(1^2 + 1^2 + 1^2) = sqrt(3) ≈ 1.73205)
expected = 1.7320508075688772
assert abs(calculate_euclidean_distance(np.asarray([1, 1, 1]), np.asarray([2, 2, 2])) - expected) < 1e-9

# 3. Symmetry
assert calculate_euclidean_distance(np.asarray([1, 5]), np.asarray([2, 10])) == calculate_euclidean_distance(np.asarray([2, 10]), np.asarray([1, 5]))

print("✅ All quick checks passed successfully!")

### Example Usage
1. Generate an embedding for your search query (using our first function)
2. Apply the similarity function across the entire column
3. Sort the DataFrame to see the most relevant results at the top

In [ ]:
amazon['cosine_similarity'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_cosine_similarity(input_embd_vector, row_embedding)
)

amazon['dot_product'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_dot_product(input_embd_vector, row_embedding)
)

amazon['euclidean_distance'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_euclidean_distance(input_embd_vector, row_embedding)
)

amazon = amazon.sort_values(by='cosine_similarity', ascending=False)
# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

### Visualize Similarity distributions
Plot the histogram of similarity distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style
sns.set_theme(style="whitegrid")

# Initialize the figure
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['cosine_similarity'],
    bins=100,
    kde=False,
    color='royalblue'
)

# Add titles and labels for readability
plt.title('Distribution of Cosine Similarity Scores', fontsize=14, pad=15)
plt.xlabel('Cosine Similarity (Higher = More Similar)', fontsize=12)
plt.ylabel('Number of Products', fontsize=12)

# Display the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style
sns.set_theme(style="whitegrid")

# Initialize the figure
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['dot_product'],
    bins=100,
    kde=False,
    color='royalblue'
)

# Add titles and labels for readability
plt.title('Distribution of Dot Product', fontsize=14, pad=15)
plt.xlabel('Dot Product (Higher = More Similar)', fontsize=12)
plt.ylabel('Number of Products', fontsize=12)

# Display the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style
sns.set_theme(style="whitegrid")

# Initialize the figure
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['euclidean_distance'],
    bins=100,
    kde=False,
    color='royalblue'
)

# Add titles and labels for readability
plt.title('Distribution of Euclidean Distance', fontsize=14, pad=15)
plt.xlabel('Euclidean Distance (Lower = More Similar)', fontsize=12)
plt.ylabel('Number of Products', fontsize=12)

# Display the plot
plt.show()

### Plot the histogram of similarity distribution

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Create a sample of a 10 top most similar records
sampled_df = amazon.iloc[:20]

# Stack the individual embedding arrays into a single 2D matrix and extract input vector
embedding_matrix = np.vstack(sampled_df['amazon_embedding'].values) - input_embd_vector

# 5. Plot the heatmap
plt.figure(figsize=(16, 4)) # Wide and short layout matching your image

# Create the heatmap
sns.heatmap(
    embedding_matrix,
    cmap='RdBu_r',     # Red-Blue diverging colormap
    center=0,          # Forces 0 to be pure white
    xticklabels=False, # Removes x-axis labels for a cleaner look
    yticklabels=False, # Removes y-axis labels
    cbar=True          # Shows the color scale on the right
)

plt.title("Embedding Vectors Difference (Sorted by Similarity)", pad=15)
plt.xlabel("Embedding Dimensions")
plt.ylabel("Products (Most Similar -> Least Similar)")

plt.show()

### Remove cosine_similarity column

In [ ]:
amazon = amazon.drop(columns=["cosine_similarity"])

### Add truncated vector (MRL) 

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter
import pandas as pd

# truncate embeddings to 128 dimentions and store results as "amazon_embedding_128" column
amazon['amazon_embedding_128'] = [
    embed[:128] for embed in amazon['amazon_embedding'].tolist()
]

# L2 normalize each truncated embedding and store results as "amazon_embedding_l2norm_128" column
amazon['amazon_embedding_l2norm_128'] = [
    l2_normalize(embed) for embed in amazon['amazon_embedding_128'].tolist()
]

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

### Load second dataset
The second table contains the same information but for product information scrapped from Google website:

In [ ]:
google = pd.read_csv("../data/GoogleProducts.csv.gz")
google.columns = [
    "idGoogleBase",
    "google_title",
    "google_description",
    "google_manufacturer",
    "google_price",
]
google = google.dropna(subset=['google_description'])
google.head()

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter

# 1. Define your batch size
BATCH_SIZE = 20

# 2. Extract the descriptions as a standard Python list
#descriptions = google['google_description'].tolist()
combined_text = google['google_title'].fillna('') + " " + google['google_description'].fillna('')

# 2. Convert to list and apply the truncation logic we used earlier
MAX_CHARS = 4000
descriptions = [str(text)[:MAX_CHARS] for text in combined_text.tolist()]
all_embeddings = []

print(f"Starting to generate embeddings for {len(descriptions)} documents in batches of {BATCH_SIZE}...")

# 3. Loop through the list in chunks
for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # Slice the list to get the current batch
    batch_texts = descriptions[i : i + BATCH_SIZE]
    
    # Call your optimized embed_text function
    batch_embeddings = embed_text(batch_texts)
    
    # Safety check: if the API call failed and returned an empty list, 
    # pad with None so our final list length matches the DataFrame length.
    if not batch_embeddings:
        logging.warning(f"Batch {i} to {i + BATCH_SIZE} failed. Padding with None.")
        batch_embeddings = [None] * len(batch_texts)
        
    all_embeddings.extend(batch_embeddings)
    
    # Pause briefly to avoid hitting Requests-Per-Minute (RPM) limits.
    # Adjust this value based on your specific API quota.
    time.sleep(1.0) 

# 4. Assign the resulting embeddings back to the DataFrame
if len(all_embeddings) == len(google):
    google['google_embedding'] = all_embeddings
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(all_embeddings)} embeddings for {len(google)} rows.")



import pandas as pd

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(google.head())

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter
import pandas as pd

# truncate embeddings to 128 dimentions and store results as "google_embedding_128" column
google['google_embedding_128'] = [
    embed[:128] for embed in google['google_embedding'].tolist()
]

# L2 normalize each truncated embedding and store results as "google_embedding_l2norm_128" column
google['google_embedding_l2norm_128'] = [
    l2_normalize(embed[:128]) for embed in google['google_embedding_128'].tolist()
]

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

### Load ground-truth dataset

The last table contains a matching of product information on both website corresponding to the same product, but possibly described differently on the two websites:

We will use this dataset to to evaluate out semantic matching approach. 
It takes a sample (whose size is controlled by `SAMPLE_SIZE`) of matching product ID's in the `matching` dataframe, and it joins the information of the Google and Amazon product information contained in the `google` and `amazon` dataframes. Then it extracts pairs of matching Google and Amazon descriptions and creates a table of matching descriptions with columns `description_1` (Google), `description_2` (Amazon), and a target column named `match` whose value is set to `yes` since we have only matching pairs so far.
To create description pairs of not matching product, we permutate the second description columns while keeping the first description fixed, and concatenate this new dataframe of not matching descriptions to the one of matching description. We shuffle and then split the resulting table into two equal sized dataframes, which we save on disk as our eval and test splits. 

Observe that we only use the `title` column as product description. So there is much more info in the raw data. Nevertheless, we will see that Gemini will achieve a performance close to 100% on the test set. Remarkable!

**Note:** Uncomment the last line if you want to re-generate the eval and test set on a different sample of the data.

In [ ]:
matching = pd.read_csv("../data/Amzon_GoogleProducts_perfectMapping.csv.gz")
SAMPLE_SIZE=1000 # Using just first 1000 matched products pairs
matching = matching.merge(right=google, how="left", on="idGoogleBase")
matched_products = matching.merge(right=amazon, how="left", on="idAmazon")
matched_products = matched_products.dropna(subset=['amazon_description'])
matched_products = matched_products.dropna(subset=['google_description'])
matched_products = matched_products[:SAMPLE_SIZE]
with pd.option_context('display.max_colwidth', 100):
    display(matched_products.head())

## 📊 Evaluation & Ranking Matching Results

The **`evaluate_matching_ranked`** evaluates entity matching performance (e.g., pairing Amazon product listings with Google product listings) using an iterative pairwise comparison approach and calculates **Top-K accuracy** metrics.

---

### Key Algorithmic Concepts

#### 1. Pairwise Comparison Matrix ($O(N \times M)$ Complexity)
For a dataset containing $N$ Amazon items and $M$ Google items, the function constructs a 2D grid (matrix) of size $N \times M$. Every item in the Amazon dataset is systematically compared against every item in the Google dataset using a user-defined `similarity_func`:

$$\mathbf{S}_{i, j} = \text{similarity\_func}(\text{Amazon}_i, \text{Google}_j)$$

#### 2. Sorting Strategy
The sorting behavior adapts dynamically based on whether your metric is a **distance** (where lower values are better) or a **similarity** (where higher values are better):

* **Distance-based ($is\_distance=True$):** Ranks the indices in **ascending order**.
* **Similarity-based ($is\_distance=False$):** Ranks the indices in **descending order** (accomplished using the negative matrix negation: `-similarity_matrix`).

---

### Parameter Configuration

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`df`** | *pandas.DataFrame* | *Required* | Input DataFrame containing the parallel data streams (e.g., ground-truth pairs and identifiers). |
| **`col_amazon`** | *str* | *Required* | Name of the DataFrame column containing the Amazon text/embeddings. |
| **`col_google`** | *str* | *Required* | Name of the DataFrame column containing the Google text/embeddings. |
| **`similarity_func`** | *function* | *Required* | The scoring function used to compare items (e.g., cosine similarity, Levenshtein distance). |
| **`is_distance`** | *bool* | `False` | Determines sorting direction. Set to `True` for distance metrics (e.g., Euclidean) and `False` for similarity metrics (e.g., Dot Product). |

---

### Code Execution Breakdown

The algorithm processes the matching problem in five distinct operational steps:

| Step | Operation | Key Variables Involved | Technical Purpose |
| :--- | :--- | :--- | :--- |
| **1** | **Initialization & Setup** | `amazon_data`, `google_data`, `google_ids` | Copies the input DataFrame and extracts targets into lightweight Python lists for faster memory access during iteration. |
| **2** | **Full Iteration Matrix** | `similarity_matrix` | Executes nested loops to run `similarity_func` exactly $N \times M$ times. Time-stamped using `time.perf_counter()`. |
| **3** | **Index Ranking** | `ranked_indices` | Uses `np.argsort` along `axis=1` to rank matches. Lower values rank first for distance; higher values rank first for similarity. |
| **4** | **Row Evaluation** | `all_ranked_ids`, `correct_match_ranks` | Iterates over each Amazon row to find where the true target Google ID ranks, converting the raw array index to a **1-based Rank**. |
| **5** | **Metrics & Output Generation** | `top_1_acc` to `top_20_acc` | Computes Top-K accuracies (percentage of times the true match fell in the top $K$ choices) and outputs stats to the console. |

---

### Output Metric Interpretations

* **Best Match Accuracy (Top-1):** The percentage of instances where the model's absolute #1 recommendation was the true target match.
* **Top-K Accuracy ($K \in \{3, 5, 10, 20\}$):** The percentage of instances where the correct match was contained within the top $K$ ranked predictions:
  
$$\text{Top-}K\text{ Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{I}(\text{Rank}_i \le K) \times 100$$

*(where $\mathbb{I}$ is the indicator function returning 1 if the condition is true, and 0 otherwise).*


In [ ]:
import time
import numpy as np
import pandas as pd

def evaluate_matching_ranked(df, col_amazon, col_google, similarity_func, is_distance=False):
    """
    Evaluates matches between Amazon and Google columns and ranks all combinations.
    
    Returns:
    - result_df: DataFrame with appended ranking lists (1 to N) and Top-K metrics.
    """
    result_df = df.copy()
    
    amazon_data = result_df[col_amazon].tolist()
    google_data = result_df[col_google].tolist()
    
    # Extract the ground truth IDs to map back to our predictions
    google_ids = result_df['idGoogleBase'].values 
    
    N = len(amazon_data)
    M = len(google_data)
    
    metric_type = "distance" if is_distance else "similarity"
    print(f"Calculating {metric_type} iteratively for {N} x {M} ({N * M:,}) combinations...")
    
    similarity_matrix = np.zeros((N, M))
    start_time = time.perf_counter()

    # --- Full Iteration Logic ---
    # We compare every Amazon item to every Google item.
    for i in range(N):
        for j in range(M): 
            similarity_matrix[i, j] = similarity_func(amazon_data[i], google_data[j])
            
    end_time = time.perf_counter()

    # --- Calculate Performance Metrics ---
    execution_time = end_time - start_time
    
    # --- Ranking Logic ---
    # np.argsort returns the indices that would sort the array. 
    if is_distance:
        # For distances, lower is better (ascending sort)
        ranked_indices = np.argsort(similarity_matrix, axis=1)
    else:
        # For similarities, higher is better (descending sort)
        # We use negative similarity_matrix to sort highest-to-lowest
        ranked_indices = np.argsort(-similarity_matrix, axis=1)

    # Pre-allocate lists for our new DataFrame columns
    all_ranked_ids = []
    all_ranked_scores = []
    correct_match_ranks = []

    for i in range(N):
        # Extract the sorted indices for this specific Amazon row
        row_sorted_indices = ranked_indices[i]
        
        # Reorder the IDs and Scores based on the sorted indices
        sorted_scores = similarity_matrix[i, row_sorted_indices]
        sorted_ids = google_ids[row_sorted_indices]
        
        all_ranked_ids.append(sorted_ids.tolist())
        all_ranked_scores.append(sorted_scores.tolist())
        
        # Calculate what rank the actual correct ID achieved (1-based indexing)
        true_id = result_df['idGoogleBase'].iloc[i]
        rank_pos = np.where(sorted_ids == true_id)[0]
        
        if len(rank_pos) > 0:
            correct_match_ranks.append(rank_pos[0] + 1) # +1 so rank starts at 1, not 0
        else:
            correct_match_ranks.append(None)
            
    # --- Assigning to DataFrame ---
    result_df['ranked_predicted_idGoogleBase'] = all_ranked_ids
    result_df['ranked_scores'] = all_ranked_scores
    result_df['correct_match_rank'] = correct_match_ranks
    
    # Preserve original logic: isolate the #1 best match
    result_df['predicted_idGoogleBase'] = [ids[0] for ids in all_ranked_ids]
    result_df['best_match_score'] = [scores[0] for scores in all_ranked_scores]
    result_df['is_correct_match'] = result_df['idGoogleBase'] == result_df['predicted_idGoogleBase']
    
    # Only calculate the diagonal pair score if the matrix is perfectly square
    if N == M:
        result_df['pair_score'] = similarity_matrix.diagonal()
    else:
        result_df['pair_score'] = None
    
    total_comparisons = N * M
    
    # Calculate Top-K Accuracies
    top_1_acc = (result_df['correct_match_rank'] == 1).mean() * 100
    top_3_acc = (result_df['correct_match_rank'] <= 3).mean() * 100
    top_5_acc = (result_df['correct_match_rank'] <= 5).mean() * 100
    top_10_acc = (result_df['correct_match_rank'] <= 10).mean() * 100
    top_20_acc = (result_df['correct_match_rank'] <= 20).mean() * 100
    
    print("-" * 40)
    print(f"Execution time: {execution_time:.4f} seconds")
    print(f"Total pairwise comparisons calculated: {total_comparisons:,}")
    print(f"Average time per comparison: {(execution_time / total_comparisons) * 1e6:.2f} microseconds")
    print("-" * 40)
    print("Model Accuracy (Where did the correct match rank?):")
    print(f"Best Match Accuracy: {top_1_acc:.2f}%")
    print(f"Top-3 Accuracy: {top_3_acc:.2f}%")
    print(f"Top-5 Accuracy: {top_5_acc:.2f}%")
    print(f"Top-10 Accuracy: {top_10_acc:.2f}%")
    print(f"Top-20 Accuracy: {top_20_acc:.2f}%")
    print("-" * 40)
    
    return result_df

### Experimentation: Semantic Matching Configurations

The code systematic evaluates every combination of **embeddings** and **distance/similarity metrics** by calling the evaluation pipeline `evaluate_matching_ranked` for each pair. 

In [ ]:
import time
import pandas as pd
import numpy as np

# Define configuration setups
embd_configs = ["embedding", "embedding_128", "embedding_l2norm_128"]

# Pair the function object with a readable name and its distance property
similarity_configs = [
    {"func": calculate_euclidean_distance, "name": "Euclidean Distance", "is_distance": True},
    {"func": calculate_cosine_similarity, "name": "Cosine Similarity", "is_distance": False},
    {"func": calculate_dot_product, "name": "Dot Product", "is_distance": False}
]

# List to compile statistics for each run
benchmark_results = []

for embd_config in embd_configs:
    for sim_cfg in similarity_configs:
        sim_func = sim_cfg["func"]
        sim_name = sim_cfg["name"]
        is_distance = sim_cfg["is_distance"]
        
        config_label = f"{embd_config} + {sim_name}"
        print(f"\n[RUNNING] Configuration: {config_label}")
        
        # Track overall execution time of the call
        start_time = time.perf_counter()
        
        # Execute the original evaluation method
        evaluated_df = evaluate_matching_ranked(
            df=matched_products, 
            col_amazon=f'amazon_{embd_config}', 
            col_google=f'google_{embd_config}', 
            similarity_func=sim_func,
            is_distance=is_distance
        )
        
        total_time = time.perf_counter() - start_time
        
        # Pull performance metrics directly from the returned DataFrame
        top_1_acc = (evaluated_df['correct_match_rank'] == 1).mean() * 100
        top_3_acc = (evaluated_df['correct_match_rank'] <= 3).mean() * 100
        top_5_acc = (evaluated_df['correct_match_rank'] <= 5).mean() * 100
        top_10_acc = (evaluated_df['correct_match_rank'] <= 10).mean() * 100
        top_20_acc = (evaluated_df['correct_match_rank'] <= 20).mean() * 100
        
        # Append all metadata and metrics
        benchmark_results.append({
            "Embedding Config": embd_config,
            "Similarity Metric": sim_name,
            "Configuration": config_label,
            "Top-1 Accuracy (%)": top_1_acc,
            "Top-3 Accuracy (%)": top_3_acc,
            "Top-5 Accuracy (%)": top_5_acc,
            "Top-10 Accuracy (%)": top_10_acc,
            "Top-20 Accuracy (%)": top_20_acc,
            "Execution Time (s)": total_time
        })

# Create the final comparison DataFrame
comparison_df = pd.DataFrame(benchmark_results)
print("\n[SUCCESS] All configurations evaluated successfully!")


### 

### Visualize from Top-1 to Top-20 accuracy progression for each single embedding/similarity configuration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_all_configs_grid(df):
    """
    Plots a 3x3 grid of separate charts. Each chart shows the Top-1 to Top-20
    accuracy progression for a single embedding/similarity configuration.
    """
    sns.set_theme(style="whitegrid")
    
    # Identify unique configurations (3 embedding configs * 3 similarity configs = 9 total)
    configs = df["Configuration"].unique()
    num_configs = len(configs)
    
    # Calculate grid dimensions dynamically
    cols = 3
    rows = (num_configs + cols - 1) // cols
    
    # Initialize a clean, spacious figure layout
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows), sharey=True)
    axes = axes.flatten()  # Flatten the grid into a 1D array for easy iteration
    
    # Define metrics and their corresponding columns in the DataFrame
    metrics = ["Top-1", "Top-3", "Top-5", "Top-10", "Top-20"]
    metric_cols = [f"{m} Accuracy (%)" for m in metrics]
    
    for i, config in enumerate(configs):
        ax = axes[i]
        
        # Isolate the row belonging to this specific configuration
        config_row = df[df["Configuration"] == config]
        accuracy_values = config_row[metric_cols].values[0]
        
        # Plot a sequential bar chart (using a beautiful green-to-blue 'crest' gradient)
        # Added hue=metrics and legend=False to prevent FutureWarnings
        sns.barplot(x=metrics, y=accuracy_values, ax=ax, hue=metrics, palette="crest", legend=False)
        
        # Subplot Customization
        ax.set_title(config, fontsize=12, fontweight='bold', pad=10)
        ax.set_ylim(0, 115)
        ax.set_ylabel("Accuracy (%)" if i % cols == 0 else "")  # Y-axis label only on the left column
        
        # Changed fontweight to 'bold' to avoid 'semibold' font loading warnings
        ax.set_xlabel("Accuracy Threshold", fontsize=9, fontweight='bold')
        
        # Draw value labels directly on top of each bar
        for bar in ax.patches:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0, 
                height + 2, 
                f'{height:.1f}%', 
                ha='center', 
                va='bottom', 
                fontsize=9, 
                fontweight='bold'  # Changed fontweight to 'bold' to avoid font loading warnings
            )
            
    # Remove any extra empty subplots in case of an uneven grid size
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.suptitle("Top-K Accuracy Progression per Configuration", fontsize=20, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

# Run the grid visualizer
plot_all_configs_grid(comparison_df)

### Compare accuracy and run times across configuration combinations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_configuration_dashboard_vertical(df):
    """
    Generates a dual-plot dashboard with vertically stacked subplots 
    to compare accuracy and run times across configuration combinations.
    """
    # Apply a clean, modern aesthetic
    sns.set_theme(style="whitegrid")
    
    # Restructured to 2 rows, 1 column. 
    # Adjusted figsize to be taller (12 width x 16 height) to give stacked charts ample breathing room.
    fig, axes = plt.subplots(2, 1, figsize=(12, 16))
    
    # --- Plot 1: Multi-Metric Accuracy Comparison (Top Plot) ---
    accuracy_cols = ["Top-1 Accuracy (%)", "Top-5 Accuracy (%)", "Top-20 Accuracy (%)"]
    df_melted = df.melt(
        id_vars=["Configuration"],
        value_vars=accuracy_cols,
        var_name="Accuracy Metric",
        value_name="Accuracy (%)"
    )
    
    # This plot already uses hue cleanly!
    sns.barplot(
        ax=axes[0],
        data=df_melted,
        x="Configuration",
        y="Accuracy (%)",
        hue="Accuracy Metric",
        palette="Blues_d"
    )
    
    axes[0].set_title("Accuracy Metrics by Configuration", fontsize=16, fontweight='bold', pad=15)
    axes[0].set_xlabel("")  # Hide x-label on top plot to reduce clutter since they share labels
    axes[0].set_ylabel("Accuracy (%)", fontsize=12, fontweight='bold')
    axes[0].set_ylim(0, 115)
    axes[0].tick_params(axis='x', rotation=30, labelsize=10) # Rotated labels slightly for clean visual flow
    
    # Annotate accuracy percentages on top of the bars
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

    # --- Plot 2: Computational Speed (Bottom Plot) ---
    # Added hue="Configuration" and legend=False to prevent FutureWarnings
    sns.barplot(
        ax=axes[1],
        data=df,
        x="Configuration",
        y="Execution Time (s)",
        hue="Configuration",
        palette="Reds_d",
        legend=False
    )
    
    axes[1].set_title("Execution Time Comparison (Lower is Better)", fontsize=16, fontweight='bold', pad=15)
    axes[1].set_xlabel("Configuration Combo", fontsize=12, fontweight='bold', labelpad=15)
    axes[1].set_ylabel("Execution Time (Seconds)", fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30, labelsize=10)
    
    # Annotate seconds on top of the bars
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt='%.3fs', padding=3, fontsize=10)

    # Clean up layout and display
    plt.tight_layout()
    plt.show()

# Run the vertical visualization dashboard
plot_configuration_dashboard_vertical(comparison_df)


Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.